In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import RidgeCV
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path = "/content/drive/MyDrive/Titulacion/DatasetsFinales/"

In [ ]:
df_train = pd.read_json(path + "df_train2_feat.jsonl", orient='records', lines=True)
df_test = pd.read_json(path + "df_test2_feat.jsonl", orient='records', lines=True)

In [ ]:
df_train

In [ ]:
df_test

In [ ]:
X = df_train.iloc[:, 4:]
y = df_train['label']

In [ ]:
X

In [ ]:
"""rf = RandomForestClassifier(
    n_estimators=300,  # Más árboles para mayor estabilidad
    max_depth=5,       # Limitar profundidad para evitar sobreajuste en selección
    random_state=42,
    n_jobs=-1         # Usar todos los cores
)"""
"""ridge = RidgeCV(
    alphas=np.logspace(-4, 4, 25),  # Rango más amplio de alphas
    cv=5,
    scoring='neg_mean_squared_error'  # Para regresión
    # Para clasificación usar: scoring='accuracy'
)"""
"""clf = LogisticRegression(
    penalty='elasticnet',   # Combinación L1+L2
    solver='saga',          # Soporta elasticnet
    l1_ratio=0.5,          # Balance entre L1 y L2
    max_iter=2000,
    C=0.1,                 # Regularización más fuerte
    random_state=42
)"""
#selector = SelectFromModel(rf, threshold="1.5*mean", prefit=True)
# Paso 1: dividir datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Paso 2: Entrenar Random Forest y seleccionar features
rf = RandomForestClassifier(n_estimators=100, random_state=42)

rf.fit(X_train, y_train)

# Selección usando la media de las importancias como umbral
selector = SelectFromModel(rf, threshold="mean", prefit=True)

# Aplicar selección
X_train_sel = selector.transform(X_train)
X_test_sel = selector.transform(X_test)

print(f"Features seleccionadas: {X_train_sel.shape[1]} / {X.shape[1]}")

# Paso 3: Refinar con RidgeCV (opcional pero útil si quieres coeficientes no dispersos)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled = scaler.transform(X_test_sel)

ridge = RidgeCV(alphas=np.logspace(-3, 3, 20), cv=5)

ridge.fit(X_train_scaled, y_train)

# También puedes ver los coeficientes:
coefs = np.abs(ridge.coef_)
coef_threshold = np.percentile(coefs, 25)  # elimina el 25% más bajo
mask = coefs > coef_threshold

X_train_final = X_train_scaled[:, mask]
X_test_final = X_test_scaled[:, mask]

print(f"Features tras RidgeCV: {X_train_final.shape[1]}")

# Paso 4: Clasificador final
clf = LogisticRegression(max_iter=1000)

clf.fit(X_train_final, y_train)

# Evaluar
acc = clf.score(X_test_final, y_test)
print(f"Accuracy final: {acc:.4f}")


In [ ]:
# separar en un df las caracteristicas seleccionadas

# Obtener los nombres de las características seleccionadas
selected_feature_indices = np.where(mask)[0]
selected_feature_names = X.columns[selector.get_support()][selected_feature_indices]

# Crear un nuevo DataFrame con las características seleccionadas
df_train_selected = pd.DataFrame(X_train_final, columns=selected_feature_names)
df_test_selected = pd.DataFrame(X_test_final, columns=selected_feature_names)

# Imprimir o usar los nuevos DataFrames
print(df_train_selected.head())
print(df_test_selected.head())


In [ ]:
df_train_selected

In [ ]:
df_test_selected

In [ ]:
selected_feature_names

In [ ]:
X_new = X[selected_feature_names]

In [ ]:
X_new

In [ ]:
X_train_new = pd.concat([df_train.iloc[:, :4], X_new], axis=1)

In [ ]:
X_train_new

In [ ]:
X_test_new = pd.concat([df_test.iloc[:, :4], df_test[selected_feature_names]], axis=1)

In [ ]:
X_test_new

In [ ]:
X_train_new.to_json(path + 'df_train2_featselect2.jsonl', orient='records', lines=True, force_ascii=False)
X_test_new.to_json(path + 'df_test2_featselect2.jsonl', orient='records', lines=True, force_ascii=False)